# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FizaAslam1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. My rule and its reason codes

# Rule in plain words:
# "Score each action based on likelihood of being the best next step,
# using past behavior, product fit, and urgency."

# Reason codes dictionary
REASON_CODES = {
    'R01': 'Strong past engagement',
    'R02': 'High product affinity',
    'R03': 'Time-sensitive opportunity',
    'R04': 'Low friction to convert',
    'R05': 'Recent drop-off, re-engage',
    'R06': 'Similar segment performed well',
    'R07': 'Action matches current trend',
    'R08': 'Low cost, high return',
    'R09': 'Seasonal / cyclical fit',
    'R10': 'Default fallback'
}

# Display the rule and codes
print("=== RULE ===")
print("Score each action based on likelihood of being the best next step,")
print("using past behavior, product fit, and urgency.\n")
print("=== REASON CODES ===")
for code, desc in REASON_CODES.items():
    print(f"{code}: {desc}")

=== RULE ===
Score each action based on likelihood of being the best next step,
using past behavior, product fit, and urgency.

=== REASON CODES ===
R01: Strong past engagement
R02: High product affinity
R03: Time-sensitive opportunity
R04: Low friction to convert
R05: Recent drop-off, re-engage
R06: Similar segment performed well
R07: Action matches current trend
R08: Low cost, high return
R09: Seasonal / cyclical fit
R10: Default fallback


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Build the ranked queue (writes the CSV)

import pandas as pd
import numpy as np
from datetime import datetime
import os

# Create output directory if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Sample data - replace with your actual data
np.random.seed(42)
n_actions = 100

data = {
    'action_id': [f'A{i:03d}' for i in range(1, n_actions+1)],
    'action_name': [f'Action_{i}' for i in range(1, n_actions+1)],
    'past_engagement_score': np.random.uniform(0, 1, n_actions),
    'product_affinity_score': np.random.uniform(0, 1, n_actions),
    'urgency_score': np.random.uniform(0, 1, n_actions),
    'friction_score': np.random.uniform(0, 1, n_actions),
    'segment_performance': np.random.uniform(0, 1, n_actions),
    'trend_match': np.random.uniform(0, 1, n_actions),
    'cost_score': np.random.uniform(0, 1, n_actions),
    'seasonal_fit': np.random.uniform(0, 1, n_actions),
}

df = pd.DataFrame(data)

# Define weights for each component (sum to 1)
weights = {
    'past_engagement_score': 0.20,
    'product_affinity_score': 0.15,
    'urgency_score': 0.15,
    'friction_score': 0.10,
    'segment_performance': 0.10,
    'trend_match': 0.10,
    'cost_score': 0.10,
    'seasonal_fit': 0.10,
}

# Calculate composite score
df['score'] = sum(df[col] * weight for col, weight in weights.items())

# Assign reason codes based on highest contributing factor
def get_reason_code(row):
    # Find which component contributed most to the score
    contributions = {col: row[col] * weights[col] for col in weights.keys()}
    max_col = max(contributions, key=contributions.get)

    # Map to reason codes
    mapping = {
        'past_engagement_score': 'R01',
        'product_affinity_score': 'R02',
        'urgency_score': 'R03',
        'friction_score': 'R04',
        'segment_performance': 'R06',
        'trend_match': 'R07',
        'cost_score': 'R08',
        'seasonal_fit': 'R09',
    }
    return mapping.get(max_col, 'R10')

df['reason_code'] = df.apply(get_reason_code, axis=1)

# Rank by score (highest first)
df['rank'] = df['score'].rank(method='min', ascending=False).astype(int)

# Sort by rank
df_ranked = df.sort_values('rank').reset_index(drop=True)

# Add confidence note (based on score magnitude)
def get_confidence(score):
    if score >= 0.8:
        return 'High'
    elif score >= 0.5:
        return 'Medium'
    else:
        return 'Low'

df_ranked['confidence'] = df_ranked['score'].apply(get_confidence)

# Add "what would make it wrong" placeholder
df_ranked['what_would_make_wrong'] = 'Check user context / product availability'

# Save to CSV
output_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(output_path, index=False)
print(f"✅ Ranked queue saved to: {output_path}")
print(f"📊 Total actions ranked: {len(df_ranked)}")
print("\nTop 5 actions:")
print(df_ranked[['rank', 'action_id', 'action_name', 'score', 'reason_code', 'confidence']].head())

✅ Ranked queue saved to: work/outputs/baseline_action_score.csv
📊 Total actions ranked: 100

Top 5 actions:
   rank action_id action_name     score reason_code confidence
0     1      A079   Action_79  0.714037         R02     Medium
1     2      A013   Action_13  0.704554         R01     Medium
2     3      A097   Action_97  0.661976         R03     Medium
3     4      A046   Action_46  0.648422         R03     Medium
4     5      A021   Action_21  0.647908         R01     Medium


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Top-20 review

print("=" * 80)
print("TOP-20 ACTION REVIEW")
print("=" * 80)

top_20 = df_ranked[df_ranked['rank'] <= 20].copy()

# Add more detailed "what would make it wrong" based on reason code
wrong_conditions = {
    'R01': 'User has disengaged recently OR past data is stale',
    'R02': 'Product is out of stock OR user showed disinterest',
    'R03': 'Offer expired OR timing is now wrong',
    'R04': 'User has higher friction than expected OR device/context changed',
    'R06': 'Segment behavior changed OR sample was too small',
    'R07': 'Trend reversed OR data is outdated',
    'R08': 'Cost increased OR hidden costs emerged',
    'R09': 'Season ended OR cyclical pattern broke',
    'R10': 'No specific signal - default fallback may be wrong'
}

top_20['what_would_make_wrong'] = top_20['reason_code'].map(wrong_conditions)

# Display top 20 with all details
for idx, row in top_20.iterrows():
    print(f"\n--- Rank #{row['rank']} ---")
    print(f"Action: {row['action_name']} (ID: {row['action_id']})")
    print(f"Score: {row['score']:.4f}")
    print(f"Reason Code: {row['reason_code']} - {REASON_CODES[row['reason_code']]}")
    print(f"Confidence: {row['confidence']}")
    print(f"Would be wrong if: {row['what_would_make_wrong']}")
    print("-" * 50)

# Also save top-20 to CSV
top_20.to_csv('work/outputs/top_20_review.csv', index=False)
print("\n✅ Top-20 review saved to: work/outputs/top_20_review.csv")

TOP-20 ACTION REVIEW

--- Rank #1 ---
Action: Action_79 (ID: A079)
Score: 0.7140
Reason Code: R02 - High product affinity
Confidence: Medium
Would be wrong if: Product is out of stock OR user showed disinterest
--------------------------------------------------

--- Rank #2 ---
Action: Action_13 (ID: A013)
Score: 0.7046
Reason Code: R01 - Strong past engagement
Confidence: Medium
Would be wrong if: User has disengaged recently OR past data is stale
--------------------------------------------------

--- Rank #3 ---
Action: Action_97 (ID: A097)
Score: 0.6620
Reason Code: R03 - Time-sensitive opportunity
Confidence: Medium
Would be wrong if: Offer expired OR timing is now wrong
--------------------------------------------------

--- Rank #4 ---
Action: Action_46 (ID: A046)
Score: 0.6484
Reason Code: R03 - Time-sensitive opportunity
Confidence: Medium
Would be wrong if: Offer expired OR timing is now wrong
--------------------------------------------------

--- Rank #5 ---
Action: Action_

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Weak picks + leakage check

print("\n" + "=" * 80)
print("WEAK PICKS + LEAKAGE CHECK")
print("=" * 80)

# Identify weak picks: high rank but low score, or inconsistencies
df_ranked['score_zscore'] = (df_ranked['score'] - df_ranked['score'].mean()) / df_ranked['score'].std()
df_ranked['is_outlier'] = df_ranked['score_zscore'].abs() > 1.5

# Weak picks = actions in top 30 that are outliers (unexpectedly high or low)
weak_picks = df_ranked[(df_ranked['rank'] <= 30) & (df_ranked['is_outlier'])]

print(f"\n🔍 Found {len(weak_picks)} weak picks in top 30:")

for idx, row in weak_picks.iterrows():
    print(f"\n--- Rank #{row['rank']} (Score: {row['score']:.4f}, z: {row['score_zscore']:.2f}) ---")
    print(f"Action: {row['action_name']}")
    print(f"Reason Code: {row['reason_code']}")
    print(f"Why it looks wrong: Score is statistically unusual compared to peers")

# ---- LEAKAGE CHECK ----

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

# Check for future window leakage
# Assumption: We have columns that might contain future info
leakage_checks = {
    'future_conversion': 'Contains data from after decision date',
    'next_month_sales': 'Uses future sales data',
    'post_campaign_metrics': 'Uses metrics only available after action',
    'future_engagement': 'Uses engagement that happened later',
    'product_launch_date': 'Product not yet available at decision time',
    'competitor_response': 'Competitor action happened after',
}

# Simulate checking for leakage - replace with actual column checks
potential_leakage_cols = ['future_conversion', 'next_month_sales', 'post_campaign_metrics']
actual_cols = df.columns.tolist()

leakage_found = [col for col in potential_leakage_cols if col in actual_cols]

if leakage_found:
    print("⚠️  WARNING: Potential leakage detected!")
    print(f"These columns appear to contain future information: {leakage_found}")
    print("Recommendation: Remove or lag these columns before scoring.")
else:
    print("✅ No obvious future-window leakage detected in column names.")

# Product flag check
product_flags = ['product_in_stock', 'product_available', 'inventory_level', 'product_active']
product_flags_found = [col for col in product_flags if col in actual_cols]

if product_flags_found:
    print(f"\nℹ️  Product-related columns found: {product_flags_found}")
    print("   Ensure these are available at decision time, not after.")
else:
    print("\n✅ No product flags detected (or none that match typical names).")

# Confidence summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Total actions ranked: {len(df_ranked)}")
print(f"Top 20 reviewed: 20")
print(f"Weak picks identified: {len(weak_picks)}")
print(f"Leakage concerns: {'YES' if leakage_found else 'NO'}")
print("\n✅ All checks complete. Results saved to work/outputs/")

# Save weak picks report
weak_picks.to_csv('work/outputs/weak_picks_report.csv', index=False)
print("📁 Weak picks report saved to: work/outputs/weak_picks_report.csv")


WEAK PICKS + LEAKAGE CHECK

🔍 Found 6 weak picks in top 30:

--- Rank #1 (Score: 0.7140, z: 2.26) ---
Action: Action_79
Reason Code: R02
Why it looks wrong: Score is statistically unusual compared to peers

--- Rank #2 (Score: 0.7046, z: 2.16) ---
Action: Action_13
Reason Code: R01
Why it looks wrong: Score is statistically unusual compared to peers

--- Rank #3 (Score: 0.6620, z: 1.72) ---
Action: Action_97
Reason Code: R03
Why it looks wrong: Score is statistically unusual compared to peers

--- Rank #4 (Score: 0.6484, z: 1.58) ---
Action: Action_46
Reason Code: R03
Why it looks wrong: Score is statistically unusual compared to peers

--- Rank #5 (Score: 0.6479, z: 1.58) ---
Action: Action_21
Reason Code: R01
Why it looks wrong: Score is statistically unusual compared to peers

--- Rank #6 (Score: 0.6459, z: 1.55) ---
Action: Action_74
Reason Code: R01
Why it looks wrong: Score is statistically unusual compared to peers

LEAKAGE CHECK
✅ No obvious future-window leakage detected in c

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.